In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
                    .appName("reading_late_streaming_events") \
                    .master("local[*]") \
                    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2") \
                    .enableHiveSupport() \
                    .getOrCreate()

In [3]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, StringType

kafka_bootstrap_servers = "kafka1:19092,kafka2:19093,kafka3:19094"
events_topic = "events"
fact_table = "looker_ecommerce_dwh.fact_events"

on_time_minutes = 1        
watermark_minutes = 60     
recover_after_hours = 2   

event_schema = StructType([
    StructField("event_id", LongType(), True), 
    StructField("user_id", IntegerType(), True),
    StructField("sequence_number", IntegerType(), True), 
    StructField("session_id", StringType(), True),
    StructField("created_at", StringType(), True), 
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),   
    StructField("browser", StringType(), True),
    StructField("traffic_source", StringType(), True), 
    StructField("uri", StringType(), True),
    StructField("event_type", StringType(), True),
])

Reading the events topic

In [4]:
import pyspark.sql.functions as F

raw = spark.read.format("kafka") \
       .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
       .option("subscribe", events_topic) \
       .option("startingOffsets", "earliest").option("endingOffsets", "latest") \
       .load() \
       .select(F.col("value").cast("string").alias("value"), F.col("timestamp").alias("arrived_at"))

events = raw.groupBy("value").agg(F.min("arrived_at").alias("arrived_at"))

In [5]:
print(f"Messages in {events_topic}: {raw.count()} | unique: {events.count()}")

Messages in events: 6222 | unique: 5962


How late is each event?

In [6]:
parsed = events.select("value", "arrived_at", F.from_json("value", event_schema).alias("e"))
created_ts = F.to_timestamp("e.created_at")

valid = parsed.where(
    F.col("e.event_id").isNotNull() & (F.col("e.sequence_number") >= 1) & created_ts.isNotNull()
    & (created_ts <= F.col("arrived_at") + F.expr("INTERVAL 1 MINUTE")))

delay_minutes = (F.unix_timestamp("arrived_at") - F.unix_timestamp(created_ts)) / 60

timed = (valid
    .withColumn("delay_minutes", F.round(delay_minutes, 1))
    .withColumn("kind",
        F.when(F.col("delay_minutes") < on_time_minutes, "on time")
         .when(F.col("delay_minutes") <= watermark_minutes, "late - accepted by the stream")
         .otherwise("very late - dropped by the stream"))
    .cache())

In [7]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; } </style>"))

Summary of kinds

In [8]:
total = timed.count()
(timed.groupBy("kind")
      .agg(F.count("*").alias("events"),
           F.round(F.count("*") / total * 100, 1).alias("percent"),
           F.round(F.min("delay_minutes"), 1).alias("min_delay_min"),
           F.round(F.avg("delay_minutes"), 1).alias("avg_delay_min"),
           F.round(F.max("delay_minutes"), 1).alias("max_delay_min"))
      .orderBy("min_delay_min")
      .show(truncate=False))

+---------------------------------+------+-------+-------------+-------------+-------------+
|kind                             |events|percent|min_delay_min|avg_delay_min|max_delay_min|
+---------------------------------+------+-------+-------------+-------------+-------------+
|on time                          |5354  |92.6   |0.0          |0.0          |0.0          |
|late - accepted by the stream    |260   |4.5    |1.0          |3.1          |5.0          |
|very late - dropped by the stream|170   |2.9    |62.9         |115.9        |179.7        |
+---------------------------------+------+-------+-------------+-------------+-------------+



The late events themselves (the most delayed first)

In [9]:
late = timed.where(F.col("kind") != "on time")

(late.select("kind", "delay_minutes",
             F.col("e.event_id").alias("event_id"),
             F.col("e.user_id").alias("user_id"),
             F.col("e.event_type").alias("event_type"),
             F.substring("e.created_at", 1, 19).alias("created_at"),
             F.date_format("arrived_at", "yyyy-MM-dd HH:mm:ss").alias("arrived_at"))
     .orderBy(F.desc("delay_minutes"))
     .show(truncate=False))

+---------------------------------+-------------+-------------+-------+----------+-------------------+-------------------+
|kind                             |delay_minutes|event_id     |user_id|event_type|created_at         |arrived_at         |
+---------------------------------+-------------+-------------+-------+----------+-------------------+-------------------+
|very late - dropped by the stream|179.7        |1794344031436|115    |Product   |2026-11-10T18:05:23|2026-11-10 21:05:07|
|very late - dropped by the stream|178.0        |1794344031567|111    |Home      |2026-11-10T18:07:33|2026-11-10 21:05:31|
|very late - dropped by the stream|176.5        |1794344027892|104    |Home      |2026-11-10T17:58:10|2026-11-10 20:54:41|
|very late - dropped by the stream|176.1        |1794344033121|114    |Home      |2026-11-10T18:13:54|2026-11-10 21:10:00|
|very late - dropped by the stream|175.9        |1794344030822|null   |Cart      |2026-11-10T18:07:26|2026-11-10 21:03:21|
|very late - dro

Are the late events already in fact_events?

In [10]:
stored = spark.table(fact_table).select(F.col("event_id").alias("stored_id")).distinct()

late_check = (late.join(stored, F.col("e.event_id") == F.col("stored_id"), "left")
                  .withColumn("in_fact_events", F.col("stored_id").isNotNull()))

(late_check.groupBy("kind", "in_fact_events").count()
           .orderBy("kind", "in_fact_events")
           .show(truncate=False))

+---------------------------------+--------------+-----+
|kind                             |in_fact_events|count|
+---------------------------------+--------------+-----+
|late - accepted by the stream    |true          |260  |
|very late - dropped by the stream|false         |155  |
|very late - dropped by the stream|true          |15   |
+---------------------------------+--------------+-----+



Very late events still waiting for the late_streaming_events DAG

In [11]:
waiting = (late_check
    .where(F.col("kind").startswith("very late") & ~F.col("in_fact_events"))
    .withColumn("recoverable_from", F.to_timestamp("e.created_at") + F.expr(f"INTERVAL {recover_after_hours} HOURS"))
    .withColumn("ready", F.col("recoverable_from") <= F.current_timestamp()))

waiting.groupBy("ready").count().show()

(waiting.select(F.col("e.event_id").alias("event_id"),
                F.substring("e.created_at", 1, 19).alias("created_at"),
                F.date_format("recoverable_from", "yyyy-MM-dd HH:mm:ss").alias("recoverable_from"),
                "ready")
        .orderBy("recoverable_from")
        .show(truncate=False))

+-----+-----+
|ready|count|
+-----+-----+
| true|  105|
|false|   50|
+-----+-----+

+-------------+-------------------+-------------------+-----+
|event_id     |created_at         |recoverable_from   |ready|
+-------------+-------------------+-------------------+-----+
|1794344028092|2026-11-10T18:02:23|2026-11-10 20:02:23|true |
|1794344031436|2026-11-10T18:05:23|2026-11-10 20:05:23|true |
|1794344030822|2026-11-10T18:07:26|2026-11-10 20:07:26|true |
|1794344031567|2026-11-10T18:07:33|2026-11-10 20:07:33|true |
|1794344028892|2026-11-10T18:07:36|2026-11-10 20:07:36|true |
|1794344030751|2026-11-10T18:08:46|2026-11-10 20:08:46|true |
|1794344033121|2026-11-10T18:13:54|2026-11-10 20:13:54|true |
|1794344033024|2026-11-10T18:14:01|2026-11-10 20:14:01|true |
|1794344028338|2026-11-10T18:14:10|2026-11-10 20:14:10|true |
|1794344032562|2026-11-10T18:15:59|2026-11-10 20:15:59|true |
|1794344030441|2026-11-10T18:18:44|2026-11-10 20:18:44|true |
|1794344030077|2026-11-10T18:19:39|2026-11-10 2